# HPML Lecture 8 demo (LLM-Scale Parallelism)

High-Performance Machine Learning (COMS E6998), Columbia University, Fall 2026. Dr. Kaoutar El Maghraoui

Each section names the Lecture 8 topic it goes with. Almost everything runs on a CPU, so it costs nothing.

| Part | What you run | Lecture topic | Hardware |
|---|---|---|---|
| A1 | ZeRO memory and traffic calculator | 16 bytes per parameter, ZeRO stages | anything |
| A2 | DTensor placements and the collective each change runs | DTensor | CPU, 4 processes |
| A3 | Tensor parallelism of one MLP block, two different splits | Megatron tensor parallelism | CPU, 4 processes |
| A4 | FSDP2: what each rank stores, and the collectives in one step | FSDP2 | CPU, 4 processes |
| A5 | Pipeline schedules: GPipe, 1F1B, interleaved | pipeline schedules | anything |
| B1 | Activation checkpointing: memory against time | activation checkpointing | 1 GPU (free Colab T4) |
| B2 | DDP against FSDP2 on two GPUs | ZeRO and FSDP2 | 2 GPUs (free Kaggle "GPU T4 x2") |

Where to run it. Part A runs on free Colab with no GPU at all: the processes talk through the `gloo` backend, so you see the real sharding and the real collectives, only slower. B1 needs one GPU, and Colab's free T4 is enough. B2 needs two GPUs; Kaggle's free "GPU T4 x2" accelerator has them. If you only have one GPU, skip B2: the rest of the notebook does not depend on it.

The T4 has no BF16, so the GPU parts use FP16 autocast (B1) or FP32 (B2). The ideas are the same; only the bytes per value change.

Each script goes to a file with `%%writefile` and runs under `torchrun`, one process per rank, the same way a real job starts.

In [ ]:
import torch, os, sys
print("PyTorch", torch.__version__, "| Python", sys.version.split()[0])
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("CPU cores:", os.cpu_count())

## Part A1. How much memory, how many bytes

A plain calculator, no GPUs. It uses the formulas from the slides: 16 bytes per parameter of training state, divided by p for each item that a ZeRO stage shards, and about 2Ψ values sent per GPU per step (3Ψ for ZeRO-3).

In [ ]:
%%writefile zero_calc.py
# zero_calc.py: training-state memory per GPU for DDP and ZeRO stages 1 to 3.
# Mixed precision with Adam: 2 bytes BF16 weight, 2 bytes BF16 gradient, 12 bytes FP32 master weight and Adam m, v.
import argparse

def per_gpu_gb(psi, p, stage):
    w, g, opt = 2 * psi, 2 * psi, 12 * psi
    if stage >= 1: opt /= p
    if stage >= 2: g /= p
    if stage >= 3: w /= p
    return (w + g + opt) / 1e9

def traffic_values(psi, p, stage):
    # values each GPU sends per step with ring collectives: about 2 psi for DDP, ZeRO-1, ZeRO-2; 3 psi for ZeRO-3
    half = (p - 1) / p * psi
    return 3 * half if stage == 3 else 2 * half

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--params", type=float, default=7.5e9)
    ap.add_argument("--gpus", type=int, default=64)
    a = ap.parse_args()
    print(f"Psi = {a.params/1e9:g}B parameters, p = {a.gpus} data-parallel GPUs")
    print(f"{'':8s}{'GB per GPU':>12s}{'values sent per GPU':>24s}")
    for st, name in enumerate(["DDP", "ZeRO-1", "ZeRO-2", "ZeRO-3"]):
        print(f"{name:8s}{per_gpu_gb(a.params, a.gpus, st):12.2f}{traffic_values(a.params, a.gpus, st)/1e9:20.2f} B")

In [ ]:
!python zero_calc.py                         # the ZeRO paper's example: 7.5B on 64 GPUs

**What to check:** 120, 31.4, 16.6 and 1.88 GB per GPU, the numbers in the lecture's running example. The traffic column stays the same for DDP, ZeRO-1 and ZeRO-2 and grows by half for ZeRO-3.

**Your turn**
1. Run `!python zero_calc.py --params 30e9 --gpus 16`. That is the Part 02 knowledge check. Which stage is the first to fit in 80 GB?
2. Keep 30B and find the smallest number of GPUs for which ZeRO-2 fits. Why does ZeRO-2 never get below 60 GB, however many GPUs you add?

## Part A2. DTensor: placements are collectives

Four CPU processes form a one-dimensional DeviceMesh. One 8 × 4 tensor is placed on it in different ways, and `CommDebugMode` counts the collectives PyTorch runs when the placement changes.

In [ ]:
%%writefile dtensor_demo.py
# dtensor_demo.py: DeviceMesh, DTensor placements, and the collective each change of placement runs.
# Run: torchrun --standalone --nproc_per_node=4 dtensor_demo.py      (CPU, gloo)
import torch, torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import DTensor, distribute_tensor, Shard, Replicate, Partial
from torch.distributed.tensor.debug import CommDebugMode

def say(rank, *a):
    # print rank by rank so the output does not interleave
    for r in range(dist.get_world_size()):
        if r == rank: print(f"[rank {rank}]", *a, flush=True)
        dist.barrier()

def main():
    dist.init_process_group("gloo")
    rank, p = dist.get_rank(), dist.get_world_size()
    mesh = init_device_mesh("cpu", (p,), mesh_dim_names=("dp",))
    full = torch.arange(8 * 4, dtype=torch.float32).reshape(8, 4)     # the same full tensor on every rank

    # 1) Shard(0): each rank keeps 8/p rows
    x = distribute_tensor(full, mesh, [Shard(0)])
    say(rank, "Shard(0) local shape", tuple(x.to_local().shape), "first row", x.to_local()[0].tolist())

    # 2) Shard(0) -> Replicate(): every rank needs all rows, so PyTorch runs an all-gather
    comm = CommDebugMode()
    with comm:
        y = x.redistribute(mesh, [Replicate()])
    say(rank, "Shard -> Replicate:", dict((str(k).split(".")[-1], v) for k, v in comm.get_comm_counts().items()),
        "local shape", tuple(y.to_local().shape))

    # 3) Partial(): each rank holds a partial sum, here rank + 1 times the tensor
    part = DTensor.from_local(full * (rank + 1), mesh, [Partial()])
    comm = CommDebugMode()
    with comm:
        z = part.redistribute(mesh, [Replicate()])
    say(rank, "Partial -> Replicate:", dict((str(k).split(".")[-1], v) for k, v in comm.get_comm_counts().items()),
        "z[0,1] =", z.to_local()[0, 1].item(), f"(expected {sum(range(1, p + 1)) * full[0,1].item():g})")

    # 4) Partial() -> Shard(0): a reduce-scatter. Some gloo builds do not implement it.
    try:
        comm = CommDebugMode()
        with comm:
            s = part.redistribute(mesh, [Shard(0)])
        say(rank, "Partial -> Shard:", dict((str(k).split(".")[-1], v) for k, v in comm.get_comm_counts().items()),
            "local shape", tuple(s.to_local().shape))
    except Exception as e:
        say(rank, "Partial -> Shard needs reduce-scatter, which this gloo build does not support:", type(e).__name__)
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=4 dtensor_demo.py 2>&1 | grep -v Warning

**What to check**
- With `Shard(0)` each rank holds 2 of the 8 rows.
- `Shard → Replicate` runs one all-gather, and every rank ends with all 8 rows.
- `Partial → Replicate` runs one all-reduce. Rank k held (k + 1) times the tensor, so the result is 1 + 2 + 3 + 4 = 10 times it.
- `Partial → Shard` is a reduce-scatter. Some `gloo` builds do not have it; on GPUs with NCCL it always works.

**Your turn:** change the mesh to two dimensions, `init_device_mesh("cpu", (2, 2), mesh_dim_names=("dp", "tp"))`, and place the tensor with `[Shard(0), Replicate()]`. How many rows does each rank hold now, and which ranks hold the same rows?

## Part A3. Tensor parallelism of one MLP block

The same MLP, W1 then GeLU then W2, split over 4 processes in two ways. The script checks the output against the unsplit model and counts the collectives in the forward pass.

In [ ]:
%%writefile tp_demo.py
# tp_demo.py: tensor parallelism of one MLP block with parallelize_module.
# Run: torchrun --standalone --nproc_per_node=4 tp_demo.py      (CPU, gloo)
import torch, torch.nn as nn, torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import Replicate
from torch.distributed.tensor.parallel import parallelize_module, ColwiseParallel, RowwiseParallel
from torch.distributed.tensor.debug import CommDebugMode

H = 256

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 = nn.Linear(H, 4 * H, bias=False)
        self.w2 = nn.Linear(4 * H, H, bias=False)
    def forward(self, x):
        return self.w2(torch.nn.functional.gelu(self.w1(x)))

def run(plan_name, plan, mesh, x, ref, rank):
    torch.manual_seed(0)
    m = MLP()                                   # same weights on every rank
    parallelize_module(m, mesh, plan)
    comm = CommDebugMode()
    with comm:
        out = m(x)
    out = out.full_tensor() if hasattr(out, "full_tensor") else out
    counts = {str(k).split(".")[-1]: v for k, v in comm.get_comm_counts().items()}
    if rank == 0:
        w1 = m.w1.weight.to_local().shape
        print(f"{plan_name:22s} w1 shard {tuple(w1)}  collectives {counts}  "
              f"max error vs one GPU {(out - ref).abs().max().item():.1e}", flush=True)

def main():
    dist.init_process_group("gloo")
    rank, t = dist.get_rank(), dist.get_world_size()
    mesh = init_device_mesh("cpu", (t,), mesh_dim_names=("tp",))
    torch.manual_seed(0); ref_model = MLP()
    torch.manual_seed(1); x = torch.randn(8, H)          # same input on every rank
    ref = ref_model(x)
    if rank == 0: print(f"t = {t} GPUs (processes), x is {tuple(x.shape)}, W1 is {H}x{4*H}, W2 is {4*H}x{H}\n")
    run("columns, then rows", {"w1": ColwiseParallel(), "w2": RowwiseParallel()}, mesh, x, ref, rank)
    run("columns, then columns", {"w1": ColwiseParallel(), "w2": ColwiseParallel(output_layouts=Replicate())}, mesh, x, ref, rank)
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=4 tp_demo.py 2>&1 | grep -v Warning

**What to check:** both splits give the same output as one GPU (error around 1e-6, which is rounding). Columns then rows needs one all-reduce. Columns then columns needs all-gathers, and the first one moves the hidden activation, which is 4 times wider than the output. That is why Megatron pairs columns with rows. Each W1 shard is 1/4 of the full matrix.

**Your turn:** add a third plan, `{"w1": RowwiseParallel(), "w2": ColwiseParallel(output_layouts=Replicate())}`, and run it. Which collective appears after W1, and why must it run before the GeLU?

## Part A4. FSDP2 on CPU: one shard per rank

`fully_shard` on each block of an 8-block model, on 4 processes. The script prints how many bytes of weights and Adam state each rank keeps, and counts the all-gathers and reduce-scatters in one training step.

In [ ]:
%%writefile fsdp_cpu.py
# fsdp_cpu.py: what FSDP2 stores on each rank, and the collectives one training step runs.
# Run: torchrun --standalone --nproc_per_node=4 fsdp_cpu.py      (CPU, gloo)
import torch, torch.nn as nn, torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard

COUNT = {}
def _count(name):
    f = getattr(dist, name)
    def wrapped(*a, **k):
        COUNT[name] = COUNT.get(name, 0) + 1
        return f(*a, **k)
    setattr(dist, name, wrapped)
for n in ["all_gather_into_tensor", "reduce_scatter_tensor", "all_reduce"]:
    _count(n)

def make(layers=8, h=512):
    torch.manual_seed(0)
    return nn.Sequential(*[nn.Sequential(nn.Linear(h, h), nn.GELU()) for _ in range(layers)])

def local_bytes(tensors):
    total = 0
    for t in tensors:
        t = t.to_local() if hasattr(t, "to_local") else t
        total += t.numel() * t.element_size()
    return total

def main():
    dist.init_process_group("gloo")
    rank, p = dist.get_rank(), dist.get_world_size()
    model = make()
    n = sum(q.numel() for q in model.parameters())
    if rank == 0: print(f"{n/1e6:.2f}M parameters, {p} ranks, FP32 weights and Adam\n")
    full = local_bytes(model.parameters())

    mesh = init_device_mesh("cpu", (p,))
    for block in model:
        fully_shard(block, mesh=mesh)          # one FSDP unit per block
    fully_shard(model, mesh=mesh)
    shard = local_bytes(model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randn(32, 512)
    try:
        COUNT.clear()
        loss = model(x).pow(2).mean()
        loss.backward()
        opt.step()
        state = local_bytes([v for s in opt.state.values() for v in s.values() if torch.is_tensor(v) and v.dim() > 0])
        if rank == 0:
            print(f"weights on this rank: {shard/1e6:6.2f} MB  (a full copy is {full/1e6:.2f} MB, ratio {full/shard:.1f})")
            print(f"Adam state on this rank: {state/1e6:6.2f} MB  (a full copy would be {2*full/1e6:.2f} MB)")
            print(f"collectives in one step: {COUNT if COUNT else '(not visible from Python in this PyTorch build; see the B2 trace)'}")
            print("8 blocks: expect about 8 all-gathers in forward, 8 in backward and 8 reduce-scatters")
            print("(the count can be lower if FSDP2 keeps the last block gathered between forward and backward)")
    except Exception as e:
        if rank == 0:
            print(f"weights on this rank: {shard/1e6:6.2f} MB  (a full copy is {full/1e6:.2f} MB, ratio {full/shard:.1f})")
            print(f"The training step did not run on CPU with this PyTorch build ({type(e).__name__}: {str(e)[:120]}).")
            print("The sharding above is still real. Part B runs the full step on two GPUs.")
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=4 fsdp_cpu.py 2>&1 | grep -v Warning

**What to check**
- Each rank keeps a quarter of the weights and a quarter of the Adam state: ZeRO-3 memory, 16Ψ/p.
- In one step: one all-gather per block in forward, one per block again in backward, and one reduce-scatter per block for the gradients. These are the two halves of all-reduce from Lecture 7, used separately.
- If your PyTorch build cannot run the step on CPU, the script says so and still shows the sharding. Part B2 runs the full step on GPUs.

**Your turn:** pass `reshard_after_forward=False` to the `fully_shard` calls on the blocks and rerun. Which count drops, and what does that cost in memory?

## Part A5. Pipeline schedules

A small simulator of the three schedules from the lecture's schedules table, with forward taking 1 time unit and backward 2. Each row is a GPU: `F` is forward, `b` is backward, a blank is the bubble. It prints the measured idle fraction next to the formula.

In [ ]:
%%writefile pipe_sim.py
# pipe_sim.py: pipeline schedules on p stages and m micro-batches.
# Forward takes 1 time unit per stage, backward 2. Prints a timeline per GPU and the idle fraction.
import argparse

def orders(p, m, kind, v):
    O = []
    for r in range(p):
        L = []
        if kind == "gpipe":
            L = [("F", i, r) for i in range(m)] + [("B", i, r) for i in range(m)]
        elif kind == "1f1b":
            w = min(p - r - 1, m); f = b = 0
            for f in range(w): L.append(("F", f, r))
            f = w
            while f < m: L.append(("F", f, r)); f += 1; L.append(("B", b, r)); b += 1
            while b < m: L.append(("B", b, r)); b += 1
        else:  # interleaved 1F1B with v chunks per GPU (needs m to be a multiple of p)
            tot = m * v
            fk = lambda k: ("F", (k // (p * v)) * p + k % p, ((k // p) % v) * p + r)
            bk = lambda k: ("B", (k // (p * v)) * p + k % p, (v - 1 - (k // p) % v) * p + r)
            w = min((p - r - 1) * 2 + (v - 1) * p, tot); f = b = 0
            for f in range(w): L.append(fk(f))
            f = w
            while f < tot: L.append(fk(f)); f += 1; L.append(bk(b)); b += 1
            while b < tot: L.append(bk(b)); b += 1
        O.append(L)
    return O

def simulate(p, m, kind, v=2):
    if kind != "inter": v = 1
    S, O = p * v, orders(p, m, kind, v)
    ptr, free, done, tl = [0] * p, [0.0] * p, {}, [[] for _ in range(p)]
    live, peak = [0] * p, [0] * p
    left = sum(len(x) for x in O)
    while left:
        best = None
        for r in range(p):
            if ptr[r] >= len(O[r]): continue
            o, i, j = O[r][ptr[r]]
            if o == "F": dep = done.get(("F", i, j - 1)) if j > 0 else 0.0
            else: dep = done.get(("B", i, j + 1)) if j < S - 1 else done.get(("F", i, j))
            if dep is None: continue
            st = max(dep, free[r])
            if best is None or st < best[0]: best = (st, r)
        st, r = best
        o, i, j = O[r][ptr[r]]; ptr[r] += 1
        en = st + (1 / v if o == "F" else 2 / v)
        done[(o, i, j)] = en; free[r] = en; tl[r].append((st, en, o, i)); left -= 1
        live[r] += 1 if o == "F" else -1; peak[r] = max(peak[r], live[r])
    return max(free), tl, peak[0] / v

def show(p, m, kind, cols=96):
    T, tl, peak = simulate(p, m, kind)
    sc = cols / T
    print(f"\n{kind}: p = {p}, m = {m}")
    for r in range(p):
        row = [" "] * cols
        for st, en, o, i in tl[r]:
            ch = "F" if o == "F" else "b"
            for c in range(int(st * sc), max(int(st * sc) + 1, int(en * sc))):
                if c < cols: row[c] = ch
        print(f"GPU {r} |" + "".join(row) + "|")
    idle = 1 - 3 * m / T
    v = 2 if kind == "inter" else 1
    formula = (p - 1) / (v * m + p - 1)
    print(f"total time {T:.1f} units, idle {idle:.1%} (formula {formula:.1%}), GPU 0 holds up to {peak:g} micro-batches of activations")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--p", type=int, default=4)
    ap.add_argument("--m", type=int, default=8)
    a = ap.parse_args()
    for k in ["gpipe", "1f1b", "inter"]:
        show(a.p, a.m, k)

In [ ]:
!python pipe_sim.py --p 4 --m 8

**What to check:** GPipe and 1F1B have the same idle fraction, (p − 1)/(m + p − 1) = 27.3%, but GPU 0 holds 8 micro-batches of activations under GPipe and only 4 under 1F1B. Interleaved 1F1B with 2 chunks per GPU cuts the idle fraction to 15.8%.

**Your turn**
1. Run `--p 8 --m 8`, then `--p 8 --m 64`. Compare with the numbers in the lecture.
2. What is the smallest m that keeps 8 stages below 10% idle? Check your answer to the Part 04 knowledge check.

## Part B1. Activation checkpointing on one GPU

12 transformer layers, h = 1,024, 1,024 tokens, batch 8, FP16 autocast. The same model runs twice, without and with `torch.utils.checkpoint` around each layer. Peak memory comes from `torch.cuda.max_memory_allocated`, and time from CUDA events with a `synchronize` before reading them.

Needs a GPU: in Colab choose Runtime, Change runtime type, T4 GPU.

In [ ]:
%%writefile ckpt_gpu.py
# ckpt_gpu.py: activation checkpointing on one GPU. Free Colab T4 is enough.
# Measures peak memory and step time with and without torch.utils.checkpoint.
import argparse, torch, torch.nn as nn
from torch.utils.checkpoint import checkpoint

class Net(nn.Module):
    def __init__(self, layers, h, heads, ckpt):
        super().__init__()
        self.ckpt = ckpt
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(h, heads, 4 * h, dropout=0.0, batch_first=True, norm_first=True)
                                     for _ in range(layers)])
    def forward(self, x):
        for layer in self.layers:
            # use_reentrant=False is the recommended checkpoint mode
            x = checkpoint(layer, x, use_reentrant=False) if self.ckpt else layer(x)
        return x

def run(ckpt, a):
    torch.manual_seed(0)
    dev = "cuda"
    model = Net(a.layers, a.h, a.heads, ckpt).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randn(a.batch, a.seq, a.h, device=dev)
    def step():
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(x).float().pow(2).mean()
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    for _ in range(3): step()                       # warm-up: allocator, kernels, Adam state
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    start, stop = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(a.steps): step()
    stop.record(); torch.cuda.synchronize()        # wait for the GPU before reading the timer
    ms = start.elapsed_time(stop) / a.steps
    peak = torch.cuda.max_memory_allocated() / 1e9
    del model, opt, x; torch.cuda.empty_cache()
    return ms, peak

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--layers", type=int, default=12); ap.add_argument("--h", type=int, default=1024)
    ap.add_argument("--heads", type=int, default=16); ap.add_argument("--seq", type=int, default=1024)
    ap.add_argument("--batch", type=int, default=8); ap.add_argument("--steps", type=int, default=5)
    a = ap.parse_args()
    print(torch.cuda.get_device_name(0), f"| {a.layers} layers, h = {a.h}, s = {a.seq}, b = {a.batch}, FP16 autocast")
    base_ms, base_gb = run(False, a)
    ck_ms, ck_gb = run(True, a)
    print(f"no checkpointing:   {base_ms:7.1f} ms per step, peak {base_gb:5.2f} GB")
    print(f"full checkpointing: {ck_ms:7.1f} ms per step, peak {ck_gb:5.2f} GB")
    print(f"memory saved {base_gb - ck_gb:.2f} GB ({1 - ck_gb / base_gb:.0%}), time added {ck_ms / base_ms - 1:.0%} (the lecture's bound for the blocks is about 33%)")

In [ ]:
import torch
if torch.cuda.is_available():
    !python ckpt_gpu.py
else:
    print("No GPU here. Switch the runtime to a T4 GPU to run B1.")

**What to check:** checkpointing cuts the peak memory by several gigabytes, and the step gets slower by roughly a third, the cost of one extra forward pass. The exact numbers depend on the GPU and on PyTorch's attention kernel.

**Your turn:** double the sequence length with `--seq 2048`. Which run grows more in memory, and why? If the run without checkpointing runs out of memory, that is the point of checkpointing.

## Part B2. DDP against FSDP2 on two GPUs

The same 16-layer model (about 200M parameters, FP32 with AdamW) trained three ways: DDP, FSDP2 with `reshard_after_forward=True` (ZeRO-3 behavior) and FSDP2 with `reshard_after_forward=False` (ZeRO-2 behavior). For each, the script prints the memory each GPU holds between steps, the peak during a step, and the step time of the slowest rank.

Needs two GPUs: on Kaggle, Settings, Accelerator, "GPU T4 x2".

In [ ]:
%%writefile fsdp_gpu.py
# fsdp_gpu.py: DDP against FSDP2 on 2 GPUs: memory per GPU and step time.
# Run on Kaggle (accelerator "GPU T4 x2", free):  torchrun --standalone --nproc_per_node=2 fsdp_gpu.py
import argparse, os, torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard

def make(layers, h):
    torch.manual_seed(0)
    return nn.Sequential(*[nn.TransformerEncoderLayer(h, 16, 4 * h, dropout=0.0, batch_first=True, norm_first=True) for _ in range(layers)])

def measure(kind, a, rank, local, p):
    dev = torch.device("cuda", local)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = make(a.layers, a.h).to(dev)                     # FP32 weights, so the state is 4 + 4 + 8 = 16 bytes per parameter
    if kind == "DDP":
        model = DDP(model, device_ids=[local])
    else:
        mesh = init_device_mesh("cuda", (p,))
        for block in model:
            fully_shard(block, mesh=mesh, reshard_after_forward=(kind == "FSDP2 (ZeRO-3)"))
        fully_shard(model, mesh=mesh)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randn(a.batch, a.seq, a.h, device=dev)
    def step():
        loss = model(x).pow(2).mean(); loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    for _ in range(3): step()                                # warm-up, creates Adam state
    torch.cuda.synchronize(); dist.barrier(); torch.cuda.reset_peak_memory_stats()
    state = torch.cuda.memory_allocated(dev) / 1e9            # weights + Adam state held between steps
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(a.steps): step()
    e.record(); torch.cuda.synchronize()
    ms = torch.tensor([s.elapsed_time(e) / a.steps], device=dev)
    dist.all_reduce(ms, op=dist.ReduceOp.MAX)                 # the slowest rank sets the step time
    peak = torch.cuda.max_memory_allocated(dev) / 1e9
    del model, opt, x; torch.cuda.empty_cache()
    return state, peak, ms.item()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--layers", type=int, default=16); ap.add_argument("--h", type=int, default=1024)
    ap.add_argument("--seq", type=int, default=512); ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--steps", type=int, default=5)
    a = ap.parse_args()
    dist.init_process_group("nccl")
    rank, p, local = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local)
    n = sum(q.numel() for q in make(a.layers, a.h).parameters())
    if rank == 0:
        print(f"{torch.cuda.get_device_name(local)} x {p} | {n/1e6:.0f}M parameters, FP32 + AdamW: {16*n/1e9:.2f} GB of training state per full copy")
        print(f"{'':18s}{'state GB':>10s}{'peak GB':>10s}{'ms/step':>10s}")
    for kind in ["DDP", "FSDP2 (ZeRO-3)", "FSDP2 (ZeRO-2)"]:
        state, peak, ms = measure(kind, a, rank, local, p)
        if rank == 0: print(f"{kind:18s}{state:10.2f}{peak:10.2f}{ms:10.1f}", flush=True)
    if rank == 0:
        print(f"\nBetween steps the gradients are freed, so 'state' is weights + Adam = 12 bytes per parameter:")
        print(f"model says DDP {12*n/1e9:.2f} GB per GPU, FSDP2 {12*n/1e9/p:.2f} GB per GPU. Peak adds gradients and activations.")
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
import torch
if torch.cuda.device_count() >= 2:
    !torchrun --standalone --nproc_per_node=2 fsdp_gpu.py 2>&1 | grep -v Warning
else:
    print("B2 needs 2 GPUs (Kaggle: GPU T4 x2). Found", torch.cuda.device_count())

**What to check**
- The state between steps halves from DDP to FSDP2 on 2 GPUs: each GPU keeps half the weights and half the Adam state.
- FSDP2 with ZeRO-3 behavior is usually the slowest of the three: it gathers the weights again in backward. ZeRO-2 behavior saves that gather and uses more memory at peak.
- The T4s talk over PCIe, not NVLink, so communication is a large part of each step. On NVLink GPUs the gap is smaller.

**Your turn:** raise `--layers` until DDP runs out of memory on a 16 GB T4 while FSDP2 still fits. How many parameters is that, and does it match 16 bytes per parameter?

## Questions to answer before the quiz

1. A1: why do ZeRO-1 and ZeRO-2 send the same bytes as DDP?
2. A2: which collective turns a `Partial` placement into `Shard`, and where does FSDP use it?
3. A3: in "columns, then columns", why is the first all-gather 4 times larger than the second?
4. A5: 1F1B has the same bubble as GPipe. So why does everyone use 1F1B?
5. B2: you have 8 GPUs per node on NVLink and slow links between nodes. Would you run FSDP over all GPUs, or HSDP?